In [1]:
import numpy as np
import json

# Carrega o JSON
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/datas_train_mlp_v2.json', 'r') as f:
    index_data = json.load(f)

X_bruto = [] # Agora conterá apenas [Vg, Vd]
Y_labels_ponto = [] # Mantém ln(|Id|)

for item in index_data:
    npz_path = item["npz_path"]
    data = np.load(npz_path, allow_pickle=True)
    
    V_model = data["V"].ravel()
    I_model = data["I"].ravel()
    V_fixed = item["fixed_voltage_simulated"] 
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = data['meta'].tolist()['is_transfer']
    
    for V_exp, I_exp in zip(V_model, I_model):
        if is_transfer:
            V_G = V_exp
            V_D = V_fixed
        else:
            V_G = V_fixed
            V_D = V_exp
        
        # --- ADAPTAÇÃO: Lendo dados diretamente sem transformações ---
        # Passamos apenas as variáveis independentes brutas
        X_ponto = [V_G, V_D]
        
        # Label Engineering (Mantido para manter a escala logarítmica, comum em OTFTs)
        I_abs = np.abs(I_exp)
        Id_min = 1e-30 
        Y_label = np.log(max(I_abs, Id_min)) 
        
        X_bruto.append(X_ponto)
        Y_labels_ponto.append(Y_label)
        
# Conversão Final
X = np.array(X_bruto, dtype=np.float32)
Y = np.array(Y_labels_ponto, dtype=np.float32).reshape(-1, 1)

print(f"Formato de X: {X.shape}") # Resultado esperado: (n_pontos, 2)

Formato de X: (110000, 2)


In [2]:
X.shape # será (N_total_pontos, 5)
Y.shape # será (N_total_pontos, 1)


(110000, 1)

In [3]:
from sklearn.preprocessing import StandardScaler

# Padronizar X (crucial para convergência)
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

In [4]:
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(256, activation='relu', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(128, activation='relu', name='HL2'),
    Dense(64, activation='relu', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=500, batch_size=32, validation_split=0.2)

#Salvar o modelo treinado e o scaler_X
base_path = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp1"
model.save(f"{base_path}/modelo_otft_otimizado.keras")
joblib.dump(scaler_X, f"{base_path}/scaler_X.pkl")
joblib.dump(Y, f"{base_path}/scaler_Y.pkl")

2026-01-26 09:06:28.015075: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-26 09:06:28.072968: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-26 09:06:29.467414: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Epoch 1/500


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1769429190.111333 1796305 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1769429190.116498 1796305 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


2750/2750 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1632 - mae: 0.1187 - val_loss: 0.1965 - val_mae: 0.2996
Epoch 2/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0014 - mae: 0.0246 - val_loss: 0.1656 - val_mae: 0.2450
Epoch 3/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0019 - mae: 0.0247 - val_loss: 0.0859 - val_mae: 0.1742
Epoch 4/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 7.0536e-04 - mae: 0.0166 - val_loss: 0.0802 - val_mae: 0.1632
Epoch 5/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0012 - mae: 0.0184 - val_loss: 0.0424 - val_mae: 0.1317
Epoch 6/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 8.9250e-04 - mae: 0.0137 - val_loss: 0.0174 - val_mae: 0.0857
Epoch 7/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 8.3496e-04 - mae: 0.0163 - val_loss: 0.0116 - val_mae: 0.0704
Epoch 8/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 4.4873e-04 - mae: 0.0110 - val_loss: 0.0098 - val_mae: 0.0623
Epoch 9/500
2750/2750 ━━━━━━

['/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp1/scaler_Y.pkl']

In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from typing import Optional, Tuple, Callable
# Importar a função 'Callable' para indicar o tipo de argumento do modelo

def plot_curve_comparison(
    V_data: np.ndarray, 
    I_real: np.ndarray,
    V_fixed: float,
    is_transfer: bool,
    mlp_model: Optional[Tuple[Callable, object]] = None,
    yscale: str = "log", 
    title: Optional[str] = None
):
    """
    Plota a curva real (I_real vs V_data) e opcionalmente a previsão da MLP.

    Parâmetros
    ----------
    V_data : np.ndarray
        Vetor de tensão (eixo X) lido do CSV (Vg para transferência, Vd para saída).
    I_real : np.ndarray
        Vetor de corrente real (eixo Y) lido do CSV.
    V_fixed : float
        Tensão fixa da curva (Vd para transferência, Vg para saída).
    is_transfer : bool
        True se for curva de transferência (Vgs vs Ids), False se for curva de saída (Vds vs Ids).
    mlp_model : tuple (Callable, object) opcional
        Tupla contendo (função_de_preparação_X, modelo_MLP_treinado, scaler_X).
        Permite que a função gere as previsões internamente.
    yscale : str {"log", "linear", "logarithmic"}
        Tipo de escala no eixo Y.
    title : str
        Título opcional do gráfico.
    """

    fig = go.Figure()
    
    # Normaliza a escala Y para checagem
    y_scale = yscale.lower()
    is_log_scale = y_scale in ("log", "logarithmic")
    y_axis_title = "Corrente Absoluta (A)" if is_log_scale else "Corrente (A)"
    
    # -------------------------------
    # PREPARAÇÃO E PLOTAGEM DA CURVA REAL
    # -------------------------------
    
    # Aplicar a correção Logarítmica apenas para o plot, se necessário
    I_plot_real = np.abs(I_real)
    if is_log_scale:
        I_plot_real[I_plot_real <= 0] = 1e-30
    
    fig.add_trace(go.Scatter(
        x=V_data, 
        y=I_plot_real, 
        mode='lines+markers', 
        name="Real (Dados CSV)",
        line=dict(color='blue')
    ))

    # -------------------------------
    # PREVISÃO E PLOTAGEM DA MLP (Se fornecida)
    # -------------------------------
    if mlp_model is not None:
        # Desempacota as ferramentas necessárias
        prepare_X_func, model, scaler_X = mlp_model
        
        # 1. Preparar features X para a curva inteira
        X_test = prepare_X_func(V_data, V_fixed, is_transfer)
        
        # 2. Padronizar X (CRUCIAL: Usar o scaler treinado)
        X_test_scaled = scaler_X.transform(X_test)
        
        # 3. Prever ln(|Id|)
        Y_pred_ln = model.predict(X_test_scaled).ravel()
        
        # 4. Converter para Corrente (|Id|)
        I_pred_abs = np.exp(Y_pred_ln)
        
        # 5. Preparar para Plotagem Logarítmica (se necessário)
        I_plot_pred = I_pred_abs.copy()
        if is_log_scale:
            I_plot_pred[I_plot_pred <= 0] = 1e-30 
        
        fig.add_trace(go.Scatter(
            x=V_data, 
            y=I_plot_pred, 
            mode='lines', 
            name="Previsão MLP",
            line=dict(color='red', dash='dash')
        ))
        
        # Calcula MAE na escala logarítmica para referência
        I_real_ln = np.log(I_plot_real)
        I_pred_ln_for_metric = np.log(I_plot_pred)
        mae = np.mean(np.abs(I_real_ln - I_pred_ln_for_metric))
        
        if title is None:
             title = f"Curva de Teste vs. Previsão MLP (V_fixed={V_fixed}V, MAE_ln={mae:.3f})"
        else:
             title += f" (MAE_ln={mae:.3f})"


    fig.update_layout(
        title=title if title is not None else "Curva Real (CSV)",
        xaxis_title="V_G (V)" if is_transfer else "V_D (V)",
        yaxis_title=y_axis_title,
        yaxis_type="log" if is_log_scale else "linear",
        template="plotly_dark",
        legend_title="Curvas"
    )

    fig.show()
    return

In [5]:
import numpy as np

def prepare_mlp_features(V_data: np.ndarray, V_fixed: float, is_transfer: bool) -> np.ndarray:
    """
    Gera o array de features X para a MLP a partir dos vetores de tensão de uma curva.
    
    Ajustado para o novo cenário: Retorna apenas as variáveis brutas [Vg, Vd].
    """
    
    # V_data é o vetor do eixo X (V_exp). V_fixed é a tensão constante da simulação.
    
    if is_transfer:
        # Transfer: Eixo X é Vg, Tensão fixa é Vds
        V_G = V_data
        V_D = np.full_like(V_data, V_fixed)
    else: 
        # Output: Eixo X é Vds, Tensão fixa é Vg
        V_G = np.full_like(V_data, V_fixed)
        V_D = V_data

    # Construindo a matriz com apenas 2 colunas
    # Isso deve bater com o input_shape=(2,) da sua rede neural
    X_features = np.column_stack([
        V_G,
        V_D
    ])
    
    return X_features

In [19]:
    
def load_data_for_inference(csv_path: str) -> Tuple[np.ndarray, np.ndarray]:
    with open(csv_path, 'r') as f:
        infer_data = json.load(f)
    
    for item in infer_data:
        npz_path = item["npz_path"]
        csv_path = item["csv_path"]
        v_fixed = item["fixed_voltage_simulated"]
        params = item["params"]
        
        # Determinar se é curva de Transferência ou Saída
        is_transfer = 'transfer' in os.path.basename(npz_path).lower()
        
        # Carregar dados do CSV
        df_exp3 = pd.read_csv(csv_path)
        V_data = df_exp3.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
        I_data = df_exp3.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    return V_data, I_data, v_fixed, is_transfer

In [20]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model
import joblib

BASE_PATH_MODEL_EXP1 = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp1"
PATH_MODEL_EXP1 = BASE_PATH_MODEL_EXP1 + '/modelo_otft_otimizado.keras'
trained_model_exp1 = load_model(PATH_MODEL_EXP1)
scaler_X_exp1 = joblib.load(BASE_PATH_MODEL_EXP1 + "/scaler_X.pkl")
scaler_Y_exp1 = joblib.load(BASE_PATH_MODEL_EXP1 + "/scaler_Y.pkl")

In [21]:
import numpy as np
import json

PATH_TEST_JSON = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
V_real_exp1, I_real_exp1, v_fixed_exp1, is_transfer_exp1 = load_data_for_inference(PATH_TEST_JSON)

In [22]:
# 2. Preparar o pacote de ferramentas para plotagem
tools_package = (prepare_mlp_features, trained_model_exp1, scaler_X_exp1)

In [23]:
# 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
plot_curve_comparison(
    V_data=V_real_exp1, 
    I_real=I_real_exp1,
    V_fixed=v_fixed_exp1,
    is_transfer=is_transfer_exp1,
    mlp_model=tools_package,
    yscale="log",
    title=None
)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [3]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from sklearn.preprocessing import StandardScaler
import joblib

# --- 1. NORMALIZAÇÃO ADAPTADA ---
# Escalonador para as entradas [Vg, Vd]
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Escalonador para a saída ln(|Id|) -> CRUCIAL para melhorar o fit
scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y) # Y já deve ser np.log(max(abs(I), 1e-30))

# --- 2. MODELO COM ARQUITETURA APRIMORADA ---
model = Sequential([
    Input(shape=(2,)), # Entrada bruta [Vg, Vd]
    
    # Camadas mais largas e ativação 'silu' (Swish) para melhor não-linearidade
    # Dense(512, activation='silu', name='HL1'),
    Dense(256, activation='silu', name='HL2'),
    Dense(128, activation='silu', name='HL3'),
    # Dense(64, activation='silu', name='HL4'),
    
    # Dropout leve para evitar que o modelo "decore" ruídos do CSV
    Dropout(0.05),
    
    # Saída linear (vai prever o ln(|Id|) escalonado)
    Dense(1, activation='linear', name='Output')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# --- 3. TREINAMENTO ---
history = model.fit(
    X_scaled, Y_scaled, 
    epochs=1000,          # Aumentado para permitir ajuste fino
    batch_size=128, 
    validation_split=0.2,
    verbose=1
)

# --- 4. SALVAR TUDO PARA A INFERÊNCIA ---
model.save("/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp2/mlp_v2_exp2.keras")
joblib.dump(scaler_X, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp2/scaler_X.pkl")
joblib.dump(scaler_Y, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp2/scaler_Y.pkl")

print("Treinamento concluído e scalers salvos.")

2026-01-26 10:41:27.415107: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-26 10:41:27.469972: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-26 10:41:29.032211: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Epoch 1/1000


E0000 00:00:1769434890.197274 1900144 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1769434890.202003 1900144 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


688/688 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0302 - mae: 0.1144 - val_loss: 0.0425 - val_mae: 0.1289
Epoch 2/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0130 - mae: 0.0733 - val_loss: 0.0290 - val_mae: 0.1230
Epoch 3/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0105 - mae: 0.0628 - val_loss: 0.0369 - val_mae: 0.1458
Epoch 4/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0089 - mae: 0.0587 - val_loss: 0.0436 - val_mae: 0.1657
Epoch 5/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0075 - mae: 0.0542 - val_loss: 0.0414 - val_mae: 0.1618
Epoch 6/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0061 - mae: 0.0499 - val_loss: 0.0439 - val_mae: 0.1608
Epoch 7/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0047 - mae: 0.0451 - val_loss: 0.0341 - val_mae: 0.1432
Epoch 8/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0035 - mae: 0.0408 - val_loss: 0.0332 - val_mae: 0.1435
Epoch 9/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

In [5]:
def inference_with_y_scaler(v_data, v_fixed, is_transfer, model, scaler_X, scaler_Y):
    # 1. Preparar e Escalar X
    x_raw = prepare_mlp_features(v_data, v_fixed, is_transfer)
    x_scaled = scaler_X.transform(x_raw)
    # 2. Predição (Resultado ainda está na escala do scaler_Y)
    y_pred_scaled = model.predict(x_scaled, verbose=0)
    
    # 3. VOLTAR PARA O ln(|Id|) REAL
    y_pred_ln = scaler_Y.inverse_transform(y_pred_scaled).ravel()
    
    # 4. VOLTAR PARA AMPERES
    i_pred = np.exp(y_pred_ln)
    
    return i_pred

In [15]:
# # Ferramentas necessárias para a inferência completa
# # Nota: Adicionamos o scaler_Y aqui
# mlp_tools = (prepare_mlp_features, model, scaler_X, scaler_Y)

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import Optional, Tuple, Callable

def plot_curve(
    V_data: np.ndarray, 
    I_real: np.ndarray,
    V_fixed: float,
    is_transfer: bool,
    mlp_model: Optional[Tuple[Callable, object, object, object]] = None,
    yscale: str = "log", 
    title: Optional[str] = None
):
    """
    Plota a curva real e a previsão da MLP, considerando a desnormalização do Target (Y).

    mlp_model : tuple (prepare_X_func, model, scaler_X, scaler_Y)
    """

    fig = go.Figure()
    
    y_scale = yscale.lower()
    is_log_scale = y_scale in ("log", "logarithmic")
    y_axis_title = "Corrente Absoluta (A)" if is_log_scale else "Corrente (A)"
    
    # --- CURVA REAL ---
    I_plot_real = np.abs(I_real)
    if is_log_scale:
        I_plot_real[I_plot_real <= 0] = 1e-30
    
    fig.add_trace(go.Scatter(
        x=V_data, 
        y=I_plot_real, 
        mode='lines+markers', 
        name="Real (Dados CSV)",
        # marker=dict(symbol='circle-open'),
        line=dict(color='blue')
    ))

    # --- PREVISÃO MLP ---
    if mlp_model is not None:
        # 0. Desempacota as 4 ferramentas (X e Y agora têm scalers)
        prepare_X_func, model, scaler_X, scaler_Y = mlp_model
        
        # 1. Preparar features X (2 colunas: Vg, Vd)
        X_test = prepare_X_func(V_data, V_fixed, is_transfer)
        
        # 2. Padronizar X
        X_test_scaled = scaler_X.transform(X_test)
        
        # 3. Prever Y (O resultado sai escalonado entre ~ -1 e 1)
        Y_pred_scaled = model.predict(X_test_scaled, verbose=0)
        
        # 4. DESNORMALIZAR Y (Voltar para a escala do ln(|Id|))
        # O reshape(-1, 1) é necessário para o scaler do scikit-learn
        Y_pred_ln = scaler_Y.inverse_transform(Y_pred_scaled.reshape(-1, 1)).ravel()
        
        # 5. Converter de Logaritmo para Corrente Linear
        I_pred_abs = np.exp(Y_pred_ln)
        
        # 6. Preparar para Plotagem
        I_plot_pred = I_pred_abs.copy()
        if is_log_scale:
            I_plot_pred[I_plot_pred <= 0] = 1e-30 
        
        fig.add_trace(go.Scatter(
            x=V_data, 
            y=I_plot_pred, 
            mode='lines', 
            name="Previsão MLP (Optimized)",
            line=dict(color='red', width=3, dash='dash')
        ))
        
        # Métrica de Erro: MAE no domínio logarítmico
        # Usamos o real em log para comparar com o Y_pred_ln desnormalizado
        I_real_ln = np.log(np.maximum(I_plot_real, 1e-30))
        mae_ln = np.mean(np.abs(I_real_ln - Y_pred_ln))
        
        suffix = f" (MAE_ln={mae_ln:.4f})"
        title = (title if title else f"V_fixed={V_fixed}V") + suffix


    fig.update_layout(
        title=title,
        xaxis_title="V_G (V)" if is_transfer else "V_D (V)",
        yaxis_title=y_axis_title,
        yaxis_type="log" if is_log_scale else "linear",
        template="plotly_dark",
        legend_title="Curvas"
    )

    fig.show()

In [2]:
import json
import numpy as np
import pandas as pd

# 1. Escolha uma curva do seu JSON para testar
json_path = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
with open(json_path, 'r') as f:
    index_data = json.load(f)

# Vamos testar a primeira curva (índice 0)
item = index_data[0]
v_fixed = item["fixed_voltage_simulated"]
is_transfer = False # Ajuste conforme o tipo da curva

# 2. CORREÇÃO: Carregar dados de um arquivo .CSV
# Usamos pandas para ler o arquivo de texto
df_real = pd.read_csv(item["csv_path"])

# Extraímos as colunas (ajuste o nome se o seu CSV tiver cabeçalho, 
# caso contrário, usamos .iloc para pegar por posição)
V_real = df_real.iloc[:, 0].values  # Primeira coluna: Tensão
I_real = df_real.iloc[:, 1].values  # Segunda coluna: Corrente


In [3]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model
import joblib

PATH_MODEL = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v2/exp2'
trained_model = load_model(PATH_MODEL + '/mlp_v2_exp2.keras')
scaler_X = joblib.load(PATH_MODEL + "/scaler_X.pkl")
scaler_Y = joblib.load(PATH_MODEL + "/scaler_Y.pkl")

2026-01-26 12:07:08.148643: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-26 12:07:08.205607: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-26 12:07:09.893296: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1769440030.830922 2044314 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1769440030.835882 2044314 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are install

In [11]:

# 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
pacote_ferramentas = (prepare_mlp_features, trained_model, scaler_X, scaler_Y)

plot_curve(
    V_data=V_real, 
    I_real=I_real,
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    mlp_model=pacote_ferramentas,
    yscale="log",
    title="Teste de Inferência - OTFT"
)